# 房价 & 租金预测综合模型  



## 1. 导入库与全局设置 / 数据路径

In [1]:
import pandas as pd
import numpy as np
import re
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, ElasticNetCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.base import BaseEstimator, TransformerMixin

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# 全局参考日期（可按需要调整为你的比赛截止日）
REFERENCE_DATE = datetime(2025, 12, 31)

# === 请在这里改成你自己的文件名（已按你提供填写） ===
PRICE_TRAIN_PATH = "/home/mw/input/hackathon255769/ruc_Class25Q2_train_price.csv"
PRICE_TEST_PATH  = "/home/mw/input/hackathon255769/ruc_Class25Q2_test_price.csv"

RENT_TRAIN_PATH  = "/home/mw/input/hackathon255769/ruc_Class25Q2_train_rent.csv"
RENT_TEST_PATH   = "/home/mw/input/hackathon255769/ruc_Class25Q2_test_rent.csv"


## 2. 通用工具函数：户型 / 楼层 / 面积 / 朝向 / 装修 / 电梯 / 日期 / 楼型 / 付款方式 / 地理

In [2]:
# ---------- 户型 ----------
def extract_room_shape(text):
    """将“2室1厅1厨1卫”解析为 (room, hall, kitchen, bathroom)。"""
    if pd.isna(text) or text == "":
        return -1, -1, -1, -1
    matches = re.findall(r"(\d+)(室|厅|厨|卫)", str(text))
    room = hall = kitchen = bathroom = 0
    for value, unit in matches:
        value = int(value)
        if unit == "室":
            room = value
        elif unit == "厅":
            hall = value
        elif unit == "厨":
            kitchen = value
        elif unit == "卫":
            bathroom = value
    return room, hall, kitchen, bathroom


# ---------- 楼层（房价版本：底/低/中/高/顶 + 总层数） ----------
def extract_floor_info_price(text):
    """处理“低楼层(共23层) / 高楼层(共12层) / 顶层(共6层)”等。"""
    text = "" if pd.isna(text) else str(text)
    is_bottom = is_low = is_middle = is_high = is_top = 0
    total_floors = 0

    total_match = re.search(r"共(\d+)层", text)
    if total_match:
        total_floors = int(total_match.group(1))

    if "底层" in text:
        is_bottom = 1
    elif "低楼层" in text:
        is_low = 1
    elif "中楼层" in text:
        is_middle = 1
    elif "高楼层" in text:
        is_high = 1
    elif "顶层" in text:
        is_top = 1

    return is_bottom, is_low, is_middle, is_high, is_top, total_floors


# ---------- 楼层（租金版本：当前层/总层；三等分） ----------
def extract_floor_info_rent(text):
    """处理“30/45层”、“中层/7层”等形式。"""
    text = "" if pd.isna(text) else str(text)
    is_bottom = is_low = is_middle = is_high = is_top = 0
    total_floors = 0
    current_floor = 0

    # 情形一: 30/45层
    specific_floor_match = re.search(r"(\d+)/(\d+)层", text)
    if specific_floor_match:
        current_floor = int(specific_floor_match.group(1))
        total_floors = int(specific_floor_match.group(2))
        if current_floor == 1:
            is_bottom = 1
        elif current_floor == total_floors:
            is_top = 1
        else:
            third = total_floors / 3
            if current_floor <= third:
                is_low = 1
            elif current_floor <= 2 * third:
                is_middle = 1
            else:
                is_high = 1
    else:
        # 情形二: 高层/7层 等
        total_match = re.search(r"/(\d+)层", text)
        if total_match:
            total_floors = int(total_match.group(1))

        if "底层" in text:
            is_bottom = 1
        elif "低层" in text:
            is_low = 1
        elif "中层" in text:
            is_middle = 1
        elif "高层" in text:
            is_high = 1
        elif "顶层" in text:
            is_top = 1

    return is_bottom, is_low, is_middle, is_high, is_top, total_floors


# ---------- 面积 ----------
def extract_area_value(text):
    """将“86.94㎡”这类面积字段抽成数值 86.94。"""
    text = "" if pd.isna(text) else str(text)
    match = re.search(r"(\d+\.?\d*)", text)
    if match:
        return float(match.group(1))
    else:
        return 0.0


# ---------- 朝向 ----------
def extract_room_direction(text):
    """从“南 北 西 东”抽取朝向 one-hot：南/北/西/东 是否出现。"""
    text = "" if pd.isna(text) else str(text)
    south = 1 if "南" in text else 0
    north = 1 if "北" in text else 0
    west  = 1 if "西" in text else 0
    east  = 1 if "东" in text else 0
    return south, north, west, east


# ---------- 装修（房价版：精/简/其他/毛坯） ----------
def extract_room_fitment(text):
    text = "" if pd.isna(text) else str(text)
    jingzhuang = 1 if "精装" in text else 0
    jianzhuang = 1 if "简装" in text else 0
    qita       = 1 if "其他" in text else 0
    maopi      = 1 if "毛坯" in text else 0
    return jingzhuang, jianzhuang, qita, maopi


# ---------- 装修（租金版：是否精装） ----------
def extract_fitment_for_rent(text):
    text = "" if pd.isna(text) else str(text)
    return 1 if "精装" in text else 0


# ---------- 电梯 ----------
def extract_elevator_flag(text):
    """解析“有/无/1”等标记电梯：有 or 1 → 1，无 → 0。"""
    text = "" if pd.isna(text) else str(text).strip()
    if ("有" in text) or ("是" in text) or (text == "1") or ("电梯" in text and "无" not in text):
        return 1
    if "无" in text or text == "0" or "否" in text:
        return 0
    return -1


# ---------- 日期 → 相对参考日的天数 ----------
def days_since_reference(date_text, reference_date=REFERENCE_DATE):
    if pd.isna(date_text) or date_text == "":
        return np.nan
    dt = pd.to_datetime(date_text, errors="coerce")
    if pd.isna(dt):
        return np.nan
    return (reference_date - dt).days


# ---------- 小区楼型结构（塔楼/板楼/平房） ----------
def extract_structure_comm(text):
    """根据小区楼型信息，抽取塔楼/板楼/平房三类 dummy。"""
    text = "" if pd.isna(text) else str(text)
    talou = banlou = pingfang = 0
    if "塔" in text:
        talou = 1
    if "板" in text:
        banlou = 1
    if "平房" in text or ("平" in text and "平层" not in text):
        pingfang = 1
    return talou, banlou, pingfang


# ---------- 付款方式 → 月数 ----------
PAYMENT_MAPPING = {
    "月付": 1, "押一付一": 1,
    "双月付": 2, "押一付二": 2,
    "季付": 3, "押一付三": 3,
    "半年付": 6,
    "年付": 12
}

def extract_pay_months(text):
    text = "" if pd.isna(text) else str(text)
    for key, val in PAYMENT_MAPPING.items():
        if key in text:
            return float(val)
    return np.nan


# ---------- 地理坐标特征：距市中心距离 ----------
PRICE_CENTER_LON = None
PRICE_CENTER_LAT = None
RENT_CENTER_LON = None
RENT_CENTER_LAT = None

def add_geo_features_price(df: pd.DataFrame, is_train: bool) -> pd.DataFrame:
    """为房价数据添加距城市中心的距离特征。"""
    global PRICE_CENTER_LON, PRICE_CENTER_LAT
    df = df.copy()
    if ('lon' in df.columns) and ('lat' in df.columns):
        if is_train or PRICE_CENTER_LON is None:
            PRICE_CENTER_LON = df['lon'].median()
            PRICE_CENTER_LAT = df['lat'].median()
        dist = np.sqrt((df['lon'] - PRICE_CENTER_LON) ** 2 + (df['lat'] - PRICE_CENTER_LAT) ** 2)
        df['dist_to_center'] = dist.clip(0, 100)
    return df


def add_geo_features_rent(df: pd.DataFrame, is_train: bool) -> pd.DataFrame:
    """为租金数据添加距城市中心的距离特征。"""
    global RENT_CENTER_LON, RENT_CENTER_LAT
    df = df.copy()
    if ('lon' in df.columns) and ('lat' in df.columns):
        if is_train or RENT_CENTER_LON is None:
            RENT_CENTER_LON = df['lon'].median()
            RENT_CENTER_LAT = df['lat'].median()
        dist = np.sqrt((df['lon'] - RENT_CENTER_LON) ** 2 + (df['lat'] - RENT_CENTER_LAT) ** 2)
        df['dist_to_center'] = dist.clip(0, 100)
    return df


## 3. 环线编码器（严格按你原始 RingEncoder 实现）

In [3]:
class RingEncoder(BaseEstimator, TransformerMixin):
    """
    环线编码器：
    - 在给定的列中（默认 ['环线']，这里会传入 ['环线', '环线位置', '环线类别']）寻找第一个存在的；
    - 生成: '环线_missing' (是否缺失) 和 '环线_num_filled' (数值化并用均值填补)；
    - 删除原始环线列以及中间变量 '环线_num'。
    """
    def __init__(self, cols=['环线']):
        self.cols = cols
        self.mapping = {
            '内环内': 1, '二环内': 1,
            '内环至中环': 2, '二至三环': 2,
            '内环至外环': 3, '三至四环': 3,
            '中环至外环': 4, '四至五环': 4, '五至六环': 4,
            '六环外': 5, '外环外': 5
        }

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        col_exist = None
        for c in self.cols:
            if c in X.columns:
                col_exist = c
                break

        if col_exist:
            X['环线_missing'] = X[col_exist].isna().astype(int)
            X['环线_num'] = X[col_exist].map(self.mapping).astype(float)
            mean_val = np.nanmean(X['环线_num'].values)
            if np.isnan(mean_val):
                mean_val = 0.0
            X['环线_num_filled'] = X['环线_num'].fillna(mean_val)
            X = X.drop(columns=[col_exist, '环线_num'])
        else:
            X['环线_missing'] = 1
            X['环线_num_filled'] = 0.0

        return X


## 4. 房价预处理：环线 + 户型 + 楼层 + 面积 + 朝向 + 装修 + 电梯 + 日期 + 地理

In [4]:
def preprocess_price(df_raw: pd.DataFrame, is_train: bool = True) -> pd.DataFrame:
    df = df_raw.copy()

    # 删除明显无用列
    for col in ["Unnamed: 0", "序号"]:
        if col in df.columns:
            df.drop(columns=[col], inplace=True)

    # 环线编码（严格使用 RingEncoder）
    ring_enc = RingEncoder(cols=['环线', '环线位置', '环线类别'])
    df = ring_enc.transform(df)

    # 小区楼型结构（如果有）
    for struct_col in ["楼型_comm", "结构_comm", "建筑类型_comm"]:
        if struct_col in df.columns:
            tmp = df[struct_col].fillna("").apply(lambda x: pd.Series(extract_structure_comm(x)))
            tmp.columns = ["塔楼", "板楼", "平房"]
            df = pd.concat([df, tmp], axis=1)
            break

    # 户型 → 室/厅/厨/卫
    if "户型" in df.columns:
        tmp = df["户型"].fillna("").apply(lambda x: pd.Series(extract_room_shape(x)))
        tmp.columns = ["室", "厅", "厨", "卫"]
        df = pd.concat([df, tmp], axis=1)

    # 楼层（房价版）
    if "楼层" in df.columns:
        tmp = df["楼层"].fillna("").apply(lambda x: pd.Series(extract_floor_info_price(x)))
        tmp.columns = ["底层", "低楼层", "中楼层", "高楼层", "顶层", "总层数"]
        df = pd.concat([df, tmp], axis=1)

    # 建筑面积 / 套内面积
    if "建筑面积" in df.columns:
        df["建筑面积数值"] = df["建筑面积"].apply(extract_area_value)
    if "套内面积" in df.columns:
        df["套内面积数值"] = df["套内面积"].apply(extract_area_value)

    # 朝向
    if "朝向" in df.columns:
        tmp = df["朝向"].fillna("").apply(lambda x: pd.Series(extract_room_direction(x)))
        tmp.columns = ["朝南", "朝北", "朝西", "朝东"]
        df = pd.concat([df, tmp], axis=1)

    # 装修情况（精/简/其他/毛坯）
    if "装修情况" in df.columns:
        tmp = df["装修情况"].fillna("").apply(lambda x: pd.Series(extract_room_fitment(x)))
        tmp.columns = ["精装修", "简装修", "其他装修", "毛坯"]
        df = pd.concat([df, tmp], axis=1)

    # 有无电梯
    if "配备电梯" in df.columns:
        df["有无电梯"] = df["配备电梯"].fillna("").apply(extract_elevator_flag)

    # 挂牌时间 → 挂牌天数
    if "挂牌时间" in df.columns:
        df["挂牌天数"] = df["挂牌时间"].fillna("").apply(days_since_reference)

    # 地理坐标特征：距市中心距离
    df = add_geo_features_price(df, is_train=is_train)

    # 补缺
    df = df.fillna(0)
    return df


## 5. 租金预处理：环线 + 户型 + 楼层 + 面积 + 朝向 + 装修 + 电梯 + 日期 + 付款方式 + 地理

In [5]:
def preprocess_rent(df_raw: pd.DataFrame, is_train: bool = True) -> pd.DataFrame:
    df = df_raw.copy()

    for col in ["Unnamed: 0", "序号"]:
        if col in df.columns:
            df.drop(columns=[col], inplace=True)

    # 环线编码
    ring_enc = RingEncoder(cols=['环线', '环线位置', '环线类别'])
    df = ring_enc.transform(df)

    # 户型
    if "户型" in df.columns:
        tmp = df["户型"].fillna("").apply(lambda x: pd.Series(extract_room_shape(x)))
        tmp.columns = ["室", "厅", "厨", "卫"]
        df = pd.concat([df, tmp], axis=1)

    # 楼层（租金版）
    if "楼层" in df.columns:
        tmp = df["楼层"].fillna("").apply(lambda x: pd.Series(extract_floor_info_rent(x)))
        tmp.columns = ["底层", "低楼层", "中楼层", "高楼层", "顶层", "总层数"]
        df = pd.concat([df, tmp], axis=1)

    # 面积
    if "建筑面积" in df.columns:
        df["建筑面积数值"] = df["建筑面积"].apply(extract_area_value)

    # 朝向
    if "朝向" in df.columns:
        tmp = df["朝向"].fillna("").apply(lambda x: pd.Series(extract_room_direction(x)))
        tmp.columns = ["朝南", "朝北", "朝西", "朝东"]
        df = pd.concat([df, tmp], axis=1)

    # 装修：是否精装
    if "装修" in df.columns:
        df["是否精装"] = df["装修"].fillna("").apply(extract_fitment_for_rent)

    # 付款方式 → 月数
    if "付款方式" in df.columns:
        df["付款月数"] = df["付款方式"].fillna("").apply(extract_pay_months)

    # 有无电梯
    if "配备电梯" in df.columns:
        df["有无电梯"] = df["配备电梯"].fillna("").apply(extract_elevator_flag)

    # 发布日期 → 在市天数
    if "发布日期" in df.columns:
        df["在市天数"] = df["发布日期"].fillna("").apply(days_since_reference)

    # 地理坐标特征
    df = add_geo_features_rent(df, is_train=is_train)

    df = df.fillna(0)
    return df


## 6. 通用模型训练与评估：OLS / Ridge / Lasso / ElasticNet

In [6]:
def train_and_eval_models(X, y, prefix=""):
    """
    通用线性模型训练与评估：
    - 目标统一做 log1p 变换，提高数值稳定性；
    - 仅保留数值列参与建模；
    - 返回 (评估结果 DataFrame, fitted_models dict)，其中 dict: name -> (model, scaler, feature_cols)。
    """
    # 只保留数值特征
    X = X.select_dtypes(include=[np.number])
    feature_cols = X.columns.tolist()

    y_log = np.log1p(y)

    X_train, X_valid, y_train, y_valid = train_test_split(
        X, y_log, test_size=0.2, random_state=2025
    )

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_valid_s = scaler.transform(X_valid)

    models = {
        "OLS": LinearRegression(),
        "Ridge": RidgeCV(alphas=np.logspace(-3, 3, 7)),
        "Lasso": LassoCV(cv=5),
        "ElasticNet": ElasticNetCV(cv=5)
    }

    results = []
    fitted_models = {}

    for name, model in models.items():
        model.fit(X_train_s, y_train)
        y_pred_log = model.predict(X_valid_s)
        y_pred = np.expm1(y_pred_log)

        mae = mean_absolute_error(np.expm1(y_valid), y_pred)
        rmse = np.sqrt(mean_squared_error(np.expm1(y_valid), y_pred))
        r2 = r2_score(np.expm1(y_valid), y_pred)

        results.append({
            "Model": prefix + name,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2
        })

        fitted_models[name] = (model, scaler, feature_cols)

    results_df = pd.DataFrame(results)
    return results_df, fitted_models


## 7. 房价管线：导入 → 预处理 → 建模 → 评估

In [7]:
# === 7.1 导入房价数据 ===
df_price_train_raw = pd.read_csv(PRICE_TRAIN_PATH)
df_price_test_raw  = pd.read_csv(PRICE_TEST_PATH)

print("[Price] 原始训练集形状:", df_price_train_raw.shape)
print("[Price] 原始测试集形状:", df_price_test_raw.shape)

# === 7.2 预处理 ===
df_price_train = preprocess_price(df_price_train_raw, is_train=True)
df_price_test  = preprocess_price(df_price_test_raw, is_train=False)

print("[Price] 预处理后训练集形状:", df_price_train.shape)
print("[Price] 预处理后测试集形状:", df_price_test.shape)

# === 7.3 特征/目标拆分 ===
y_price = df_price_train["Price"].astype(float)
X_price_full = df_price_train.drop(columns=["Price"], errors="ignore")
X_price_test_full = df_price_test.copy()

# 对齐 train/test 特征列
common_cols_price = [c for c in X_price_full.columns if c in X_price_test_full.columns]
X_price = X_price_full[common_cols_price]
X_price_test = X_price_test_full[common_cols_price]

print("[Price] 用于建模的特征维度:", X_price.shape)

# === 7.4 训练与评估 ===
price_results, price_models = train_and_eval_models(X_price, y_price, prefix="Price_")
print("\n[Price] 各模型在验证集上的表现:")
print(price_results)

price_results.to_csv("price_model_evaluation.csv", index=False, encoding="utf-8-sig")


[Price] 原始训练集形状: (103871, 55)
[Price] 原始测试集形状: (34017, 55)
[Price] 预处理后训练集形状: (103871, 64)
[Price] 预处理后测试集形状: (34017, 64)
[Price] 用于建模的特征维度: (103871, 63)

[Price] 各模型在验证集上的表现:
              Model            MAE          RMSE        R2
0         Price_OLS  988734.481448  2.430524e+06  0.146343
1       Price_Ridge  988747.889621  2.430647e+06  0.146257
2       Price_Lasso  989968.200385  2.433366e+06  0.144346
3  Price_ElasticNet  990380.554338  2.436581e+06  0.142083


## 8. 房价：选择最优模型并在测试集上预测

In [8]:
# === 8.1 选取最佳模型 ===
best_row_price = price_results.sort_values("R2", ascending=False).iloc[0]
best_model_name_price = best_row_price["Model"].replace("Price_", "")
print(f"[Price] 最优模型: {best_model_name_price}")

best_model_price, best_scaler_price, feat_cols_price = price_models[best_model_name_price]

# === 8.2 在测试集上预测 ===
X_price_test_num = X_price_test[feat_cols_price]
X_price_test_s = best_scaler_price.transform(X_price_test_num)

y_price_pred_log = best_model_price.predict(X_price_test_s)
y_price_pred = np.expm1(y_price_pred_log)

# === 8.3 导出提交文件 ===
if "ID" in df_price_test_raw.columns:
    submission_price = pd.DataFrame({"ID": df_price_test_raw["ID"], "Price": y_price_pred})
else:
    submission_price = pd.DataFrame({"Price": y_price_pred})

submission_price.to_csv("submission_price.csv", index=False, encoding="utf-8-sig")
print("[Price] 测试集预测已保存为 submission_price.csv")

[Price] 最优模型: OLS
[Price] 测试集预测已保存为 submission_price.csv


## 9. 租金管线：导入 → 预处理 → 建模 → 评估

In [9]:
# === 9.1 导入租金数据 ===
df_rent_train_raw = pd.read_csv(RENT_TRAIN_PATH)
df_rent_test_raw  = pd.read_csv(RENT_TEST_PATH)

print("[Rent] 原始训练集形状:", df_rent_train_raw.shape)
print("[Rent] 原始测试集形状:", df_rent_test_raw.shape)

# === 9.2 预处理 ===
df_rent_train = preprocess_rent(df_rent_train_raw, is_train=True)
df_rent_test  = preprocess_rent(df_rent_test_raw, is_train=False)

print("[Rent] 预处理后训练集形状:", df_rent_train.shape)
print("[Rent] 预处理后测试集形状:", df_rent_test.shape)

# === 9.3 特征/目标拆分 ===
y_rent = df_rent_train["Price"].astype(float)   # 注意：租金任务中目标列仍为 Price
X_rent_full = df_rent_train.drop(columns=["Price"], errors="ignore")
X_rent_test_full = df_rent_test.copy()

common_cols_rent = [c for c in X_rent_full.columns if c in X_rent_test_full.columns]
X_rent = X_rent_full[common_cols_rent]
X_rent_test = X_rent_test_full[common_cols_rent]

print("[Rent] 用于建模的特征维度:", X_rent.shape)

# === 9.4 训练与评估 ===
rent_results, rent_models = train_and_eval_models(X_rent, y_rent, prefix="Rent_")
print("\n[Rent] 各模型在验证集上的表现:")
print(rent_results)

rent_results.to_csv("rent_model_evaluation.csv", index=False, encoding="utf-8-sig")


[Rent] 原始训练集形状: (98899, 46)
[Rent] 原始测试集形状: (9773, 46)
[Rent] 预处理后训练集形状: (98899, 64)
[Rent] 预处理后测试集形状: (9773, 64)
[Rent] 用于建模的特征维度: (98899, 63)

[Rent] 各模型在验证集上的表现:
             Model            MAE           RMSE        R2
0         Rent_OLS  234848.312112  492449.181554  0.340977
1       Rent_Ridge  234854.018219  492489.990736  0.340868
2       Rent_Lasso  234861.702267  492707.752362  0.340285
3  Rent_ElasticNet  234877.041363  492821.374335  0.339981


## 10. 租金：选择最优模型并在测试集上预测

In [10]:
# === 10.1 选取最佳模型 ===
best_row_rent = rent_results.sort_values("R2", ascending=False).iloc[0]
best_model_name_rent = best_row_rent["Model"].replace("Rent_", "")
print(f"[Rent] 最优模型: {best_model_name_rent}")

best_model_rent, best_scaler_rent, feat_cols_rent = rent_models[best_model_name_rent]

# === 10.2 在测试集上预测 ===
X_rent_test_num = X_rent_test[feat_cols_rent]
X_rent_test_s = best_scaler_rent.transform(X_rent_test_num)

y_rent_pred_log = best_model_rent.predict(X_rent_test_s)
y_rent_pred = np.expm1(y_rent_pred_log)

# === 10.3 导出提交文件 ===
if "ID" in df_rent_test_raw.columns:
    submission_rent = pd.DataFrame({"ID": df_rent_test_raw["ID"], "Price": y_rent_pred})
else:
    submission_rent = pd.DataFrame({"Price": y_rent_pred})

submission_rent.to_csv("submission_rent.csv", index=False, encoding="utf-8-sig")
print("[Rent] 测试集预测已保存为 submission_rent.csv")

[Rent] 最优模型: OLS
[Rent] 测试集预测已保存为 submission_rent.csv
